In [2]:
!pip install hmmlearn
import numpy as np
from hmmlearn import hmm
from sklearn.preprocessing import LabelEncoder

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.9/165.9 kB 2.6 MB/s eta 0:00:00


In [5]:
import numpy as np
from hmmlearn import hmm
from sklearn.preprocessing import LabelEncoder

# Sample text
text = "hello world hello markov model"

# Encode characters as integers
chars = sorted(list(set(text)))  # Sorted for reproducibility
encoder = LabelEncoder()
encoder.fit(chars)
encoded = encoder.transform(list(text))  # e.g., 'h' -> 0, 'e' -> 1, ...

# Reshape into column vector of shape (n_samples, 1)
X = np.array(encoded).reshape(-1, 1)

# Create HMM
model = hmm.MultinomialHMM(n_components=5, n_iter=100, random_state=42)

# Set n_features manually (must match number of unique emissions)
model.n_features = len(encoder.classes_)

# Fit model
model.fit(X)

# Predict future characters
def predict_next_chars(prefix, n_chars=10):
    # Transform prefix to encoded format
    try:
        prefix_encoded = encoder.transform(list(prefix)).reshape(-1, 1)
    except ValueError:
        raise ValueError("Prefix contains unknown characters.")

    # Decode the states
    logprob, state_sequence = model.decode(prefix_encoded, algorithm="viterbi")
    current_state = state_sequence[-1]

    output = []
    for _ in range(n_chars):
        # Choose next state based on transition probabilities
        next_state = np.random.choice(model.n_components, p=model.transmat_[current_state])
        # Choose character based on emission probabilities
        next_char_index = np.random.choice(len(encoder.classes_), p=model.emissionprob_[next_state])
        output.append(encoder.inverse_transform([next_char_index])[0])
        current_state = next_state

    return ''.join(output)

# Example
prefix = "hello "
predicted = predict_next_chars(prefix, n_chars=10)
print(f"Prefix: {prefix}\nPredicted: {predicted}")



https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340


ValueError: Unexpected number of dimensions, got 1 but expected 12